In [4]:
from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic

# 1. Initialize Spark 
# (All Iceberg, S3, and JDBC configs are automatically inherited from spark-defaults.conf!)
spark = SparkSession.builder.appName("Iceberg-Lakehouse-Demo").getOrCreate()

# Silence verbose Spark/Iceberg warnings
spark.sparkContext.setLogLevel("ERROR")


In [28]:
from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic

# Initialize Spark Session (Inherits configurations automatically from spark-defaults.conf)

# Define %%sql magic for standard Jupyter environments
@register_cell_magic
def sql(line, cell):
    # 1. Remove all single-line SQL comments (anything from '--' to the end of a line)
    cell_without_comments = re.sub(r'--.*$', '', cell, flags=re.MULTILINE)
    
    # 2. Split by semicolon to allow multiple statements per cell
    queries = [q.strip() for q in cell_without_comments.split(';') if q.strip()]
    
    for query in queries:
        df = spark.sql(query)
        # Only display output if the query returns data (skips empty DDL/DML outputs)
        if len(df.columns) > 0:
            df.show(truncate=False)
            
print("Spark Session initialized. The %%sql magic is now ready to use!")

Spark Session initialized. The %%sql magic is now ready to use!


## 1. CHECK for the CATLOGUE from POSTGRE
#### Select * from iceberg_tables
#### Get the latest metadata file being pointed

## 2. METADATA JSON
#### - Check the Metadata Data Json file from the S3 and dowlnoad and explore and Check which Snapshot version and MANIFEST LIST FILE it points

In [45]:
%%sql

SELECT * FROM demo.ecommerce.orders.metadata_log_entries order by timestamp desc;    -- metadata.json version history

SELECT * FROM demo.ecommerce.orders.history order by made_current_at DESC;  

-- s3a://warehouse/ecommerce/orders/metadata/00002-9af15b88-da53-414a-b2bf-30ae1a50371e.metadata.json

+-----------------------+--------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|timestamp              |file                                                                                              |latest_snapshot_id |latest_schema_id|latest_sequence_number|
+-----------------------+--------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|2026-11-24 04:21:02.317|s3a://warehouse/ecommerce/orders/metadata/00003-b616e04f-c9dc-4afc-84b5-ee35a52be84a.metadata.json|3281998439897304526|0               |3                     |
|2026-11-24 03:57:02.529|s3a://warehouse/ecommerce/orders/metadata/00002-9af15b88-da53-414a-b2bf-30ae1a50371e.metadata.json|5110494612978070953|0               |2                     |
|2026-11-24 01:45:28.342|s3a://warehouse/ecommerce/orders/metadata/00001-04

In [46]:
# Locate the latest metadata JSON file in the metadata folder

latest_metadata_path = "s3a://warehouse/ecommerce/orders/metadata/00003-b616e04f-c9dc-4afc-84b5-ee35a52be84a.metadata.json"

meta_df = spark.read.option("multiline", "true").json(latest_metadata_path)
meta_df.select(
    "current-snapshot-id",
    "snapshots",
    "refs"
).show(truncate=False)

+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## TWO MAIN TABLES of ICEBERG

Here is a column-by-column breakdown of `demo.ecommerce.orders.snapshots` and `demo.ecommerce.orders.history` based on the [Apache Iceberg Spark Queries Documentation](https://iceberg.apache.org/docs/latest/spark-queries/), [Conduktor's Iceberg Metadata Guide](https://www.conduktor.io/glossary/iceberg-table-architecture-metadata-and-snapshots), [IOMETE's Iceberg Reference](https://iomete.com/resources/reference/iceberg-tables/queries), and your notebook output.

---

## 1. `demo.ecommerce.orders.snapshots`

The **`snapshots`** metadata table lists all valid snapshots recorded in the table's metadata, along with when they were committed, what operation created them, which manifest list file holds their file inventory, and summary metrics.

| Column | Functionality | Use Case |
| --- | --- | --- |
| **`committed_at`** | Timestamp when the snapshot was committed to the table. | Finding the right snapshot ID for `TIMESTAMP AS OF` time-travel queries or checking when a batch/streaming write landed. |
| **`snapshot_id`** | Unique 64-bit integer ID identifying a point-in-time state of the table. | Passing a specific version to `VERSION AS OF <id>` or `snapshot-id` read options for time travel, incremental reads, or rollbacks. |
| **`parent_id`** | ID of the immediately preceding snapshot that this snapshot builds upon (`NULL` for the initial snapshot). | Tracing linear or branched snapshot chains and verifying how a new commit relates to the prior state. |
| **`operation`** | High-level write/maintenance operation that produced the snapshot (e.g., `append`, `overwrite`, `delete`, `replace`). | Auditing what changed in a commit—such as distinguishing routine `append` writes from destructive `overwrite` or `delete` jobs. |
| **`manifest_list`** | Path (e.g., `s3a://...snap-...avro`) to the **manifest list** Avro file that indexes all manifest files for this snapshot. | Inspecting the exact manifest list for a snapshot (as you did in Cell [35]) or debugging missing/corrupted snapshot metadata files. |
| **`summary`** | Map (`key -> value` strings) of commit metrics and engine properties (e.g., `added-records`, `total-records`, `added-data-files`, `total-data-files`, `spark.app.id`, `iceberg-version`). | Monitoring row/file growth, spotting small-file buildup that needs compaction, and tracing which Spark application (`spark.app.id`) wrote a snapshot. |

> **Note from your Cell [32] output:** Your initial commit (`575426542183088885`) has `parent_id = NULL`, `operation = append`, `total-records = 1000`, and `total-data-files = 3`. The second commit (`5110494612978070953`) has `parent_id = 575426542183088885`, adds `3` records in `1` file, and brings the table to `1003` total records across `4` data files.

---

## 2. `demo.ecommerce.orders.history`

The **`history`** metadata table shows the chronological log of snapshots that became the table's current state and whether each snapshot is an ancestor of the current state.

| Column | Functionality | Use Case |
| --- | --- | --- |
| **`made_current_at`** | Timestamp when that snapshot became the active/current snapshot for the table. | Auditing when a snapshot actually took effect—especially useful after rollbacks or cherry-picks where commit order and "made current" order can differ. |
| **`snapshot_id`** | ID of the snapshot that was made current. | Correlating active state transitions with `snapshots.snapshot_id` (for instance, joining `history` and `snapshots` to inspect `operation` or `summary[spark.app.id]`). |
| **`parent_id`** | ID of the parent snapshot from which this snapshot was derived (`NULL` for the first snapshot). | Reconstructing parent-child lineage across state changes. |
| **`is_current_ancestor`** | Boolean (`true` / `false`) indicating whether the snapshot lies in the ancestor lineage of the current table state. | Checking whether an older snapshot is still part of the active lineage or was bypassed/orphaned after a rollback or branch switch. |

> **Important distinction from Cell [33]:** Because snapshot `5110494612978070953` has `parent_id = 575426542183088885`, **both** rows show `is_current_ancestor = true`. That column tells you whether a snapshot is an ancestor in the current lineage, not which single row is the latest head.

---

### Quick Rule of Thumb: `snapshots` vs. `history`

* Use **`snapshots`** when you want **what each snapshot did** (`operation`, `summary` record/file counts, `manifest_list` path).
* Use **`history`** when you want **lineage and state transitions** (`made_current_at`, `parent_id`, `is_current_ancestor`).

In [47]:
%%sql

-- Check Which Snapshot ID Is Mapped to Which Snapshot / Manifest List File
-- This Give the snaphot files mapped to the snapshot ids or version and Note the Snaphot files are the manifest list files.
-- MANIFEST LIST FILE is SNAPSHOT FILE
SELECT * FROM demo.ecommerce.orders.snapshots ORDER BY committed_at DESC; 

-- every snapshot + summary

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                 |summary                                                                      

In [49]:
%%sql
SELECT * FROM demo.ecommerce.orders.history order by made_current_at;                -- snapshot lineage / parent-child


+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-11-24 01:45:28.342|575426542183088885 |NULL               |true               |
|2026-11-24 03:57:02.529|5110494612978070953|575426542183088885 |true               |
|2026-11-24 04:21:02.317|3281998439897304526|5110494612978070953|true               |
+-----------------------+-------------------+-------------------+-------------------+



## 3. MAP MANIFEST FILES FROM MANISFEST LIST FILES

In [52]:
%%sql
    
SELECT
    added_snapshot_id,
    reference_snapshot_id,
    manifest_list,
    path AS manifest_file_path,
    content,
    added_data_files_count,
    existing_data_files_count,
    deleted_data_files_count
FROM demo.ecommerce.orders.snapshots snapshot inner join demo.ecommerce.orders.all_manifests manifest
ON snapshot.snapshot_id= manifest.reference_snapshot_id
ORDER BY committed_at desc,reference_snapshot_id desc;

+-------------------+---------------------+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-------+----------------------+-------------------------+------------------------+
|added_snapshot_id  |reference_snapshot_id|manifest_list                                                                                                 |manifest_file_path                                                                    |content|added_data_files_count|existing_data_files_count|deleted_data_files_count|
+-------------------+---------------------+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-------+----------------------+-------------------------+------------------------+
|3281998439897304526|3281998

In [53]:
# Define the full S3A path to your MinIO file

## MANIFEST LIST FILE and the set of the manifest files that it points
manifest_list_file_path = "s3a://warehouse/ecommerce/orders/metadata/snap-3281998439897304526-1-7f590999-00b4-4566-acbb-fa505961a0cf.avro"

# Read the Avro file into a DataFrame
df = spark.read.format("avro").load(manifest_list_file_path)
# [span_1](start_span)[span_1](end_span)

# Print the schema (this displays all column names and data types)
df.printSchema()
# [span_2](start_span)[span_2](end_span)

# Print the output data (headers are included automatically)
print("-----------------Printing ALL Columns---------------------")
df.show(truncate=False)

print("-----------------Printing Selected Columns---------------------")

# df.select("manifest_path","manifest_length","sequence_number","added_snapshot_id","added_files_count","deleted_files_count","added_rows_count",
#           "deleted_rows_count","partitions.contains_null","partitions.lower_bound","partitions.upper_bound").orderBy("sequence_number").show(truncate=False)

df.select("manifest_path","sequence_number","added_snapshot_id","added_files_count","deleted_files_count","added_rows_count",
          "deleted_rows_count","partitions.contains_null","partitions.lower_bound","partitions.upper_bound").orderBy("sequence_number").show(truncate=False)

# [span_3](start_span)[span_3](end_span)


root
 |-- manifest_path: string (nullable = true)
 |-- manifest_length: long (nullable = true)
 |-- partition_spec_id: integer (nullable = true)
 |-- content: integer (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- min_sequence_number: long (nullable = true)
 |-- added_snapshot_id: long (nullable = true)
 |-- added_files_count: integer (nullable = true)
 |-- existing_files_count: integer (nullable = true)
 |-- deleted_files_count: integer (nullable = true)
 |-- added_rows_count: long (nullable = true)
 |-- existing_rows_count: long (nullable = true)
 |-- deleted_rows_count: long (nullable = true)
 |-- partitions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- contains_null: boolean (nullable = true)
 |    |    |-- contains_nan: boolean (nullable = true)
 |    |    |-- lower_bound: binary (nullable = true)
 |    |    |-- upper_bound: binary (nullable = true)
 |-- key_metadata: binary (nullable = true)

-----------------Printing

## EXPLAINATION of LOWER and UPPER BOUND
In your notebook, lower_bound and upper_bound in the manifest list (partitions.lower_bound / partitions.upper_bound) are stored as raw binary ([A9 50 00 00], [E2 50 00 00], [82 4E 00 00], [84 4E 00 00]), representing little-endian hexadecimal byte arrays of Apache Iceberg's serialized partition values—not plain human-readable dates or strings.

1. Why They Look Like Hex Bytes ([A9 50 00 00])
Binary encoding in Iceberg metadata: Inside Iceberg's manifest list Avro file, partitions is an array of partition summary structs (contains_null, contains_nan, lower_bound, upper_bound). Iceberg serializes partition values into binary buffers according to the partition field's underlying type.

Little-endian byte order: Multi-byte numbers in these hex arrays are typically stored in little-endian order (least significant byte first). For example, [A9 50 00 00] has bytes A9 (low byte), 50, 00, 00 (high byte), which reverses to big-endian 0x000050A9.

2. How to Interpret Them in Your Two Examples
Example A: ecommerce.orders ([A9 50 00 00] to [E2 50 00 00])
Look at the .files and .all_manifests outputs in your notebook for ecommerce.orders: the data files are partitioned by order_timestamp_day, spanning 2026-07-15 to 2026-09-10.

In Iceberg's days(...) partition transform, a day value is stored internally as an integer count of days since the Unix epoch (1970-01-01) (a 32-bit / 4-byte integer).

Converting little-endian hex to decimal days since 1970-01-01:

[A9 50 00 00] → big-endian 0x000050A9 = 20,649 days after 1970-01-01 → 2026-07-15.

[E2 50 00 00] → big-endian 0x000050E2 = 20,706 days after 1970-01-01 → 2026-09-10.


## 4. EXPLORE THE MANIFEST FILES - which points to the data files of the particular snapshot

In [54]:
# Define the full S3A path to your MinIO file
# MANIFEST FILES


# s3a://warehouse/ecommerce/orders/metadata/ccfe379b-a3e3-4c5d-bb4a-aa693ec8eae2-m0.avro
# s3a://warehouse/ecommerce/orders/metadata/c87a534b-0b49-4547-bbaa-410ae9404a30-m0.avro
# s3a://warehouse/ecommerce/orders/metadata/7f590999-00b4-4566-acbb-fa505961a0cf-m0.avro
# s3a://warehouse/ecommerce/orders/metadata/7f590999-00b4-4566-acbb-fa505961a0cf-m1.avro

manifest_file_path1 = "s3a://warehouse/ecommerce/orders/metadata/ccfe379b-a3e3-4c5d-bb4a-aa693ec8eae2-m0.avro"
manifest_file_path2 = "s3a://warehouse/ecommerce/orders/metadata/c87a534b-0b49-4547-bbaa-410ae9404a30-m0.avro"
manifest_file_path3 = "s3a://warehouse/ecommerce/orders/metadata/7f590999-00b4-4566-acbb-fa505961a0cf-m0.avro"
manifest_file_path4 = "s3a://warehouse/ecommerce/orders/metadata/7f590999-00b4-4566-acbb-fa505961a0cf-m1.avro"
# Read the Avro file into a DataFrame
df1 = spark.read.format("avro").load(manifest_file_path1)
df2 = spark.read.format("avro").load(manifest_file_path2)
df3 = spark.read.format("avro").load(manifest_file_path3)
df4 = spark.read.format("avro").load(manifest_file_path4)
# [span_1](start_span)[span_1](end_span)

# Print the schema (this displays all column names and data types)
# df1.printSchema()
# [span_2](start_span)[span_2](end_span)
cols = ["status","snapshot_id", "file_sequence_number", "data_file.file_path", "data_file.partition",
        "data_file.record_count", "data_file.file_size_in_bytes"]
# Print the output data (headers are included automatically)
df1.select(cols).orderBy("sequence_number").show(truncate=False)
df2.select(cols).orderBy("sequence_number").show(truncate=False)
df3.select(cols).orderBy("sequence_number").show(truncate=False)
df4.select(cols).orderBy("sequence_number").show(truncate=False)
# [span_3](start_span)[span_3](end_span)

# status : 1 -> Added , 0 -> Existing, 2 -> deleted.


+------+------------------+--------------------+----------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|status|snapshot_id       |file_sequence_number|file_path                                                                                                                         |partition   |record_count|file_size_in_bytes|
+------+------------------+--------------------+----------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|1     |575426542183088885|NULL                |s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-07-15/00000-11-7ed76c1c-42ea-4084-bd4a-0f5c9a538d0c-0-00001.parquet|{2026-07-15}|318         |5242              |
|1     |575426542183088885|NULL                |s3a://warehouse/ecommerce/orders/data/order_timestam

### Checking the data in the parquet files - To be check after the update operation

In [58]:
file1="s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-08-20/00000-64-8a686c50-639b-4a19-810b-df04f623bfd0-00002.parquet"
file2="s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-64-8a686c50-639b-4a19-810b-df04f623bfd0-00001.parquet"
file3="s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-08-20/00000-64-8a686c50-639b-4a19-810b-df04f623bfd0-00001-deletes.parquet"
file4="s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-64-8a686c50-639b-4a19-810b-df04f623bfd0-00002-deletes.parquet"
df1=spark.read.parquet(file1)
df2=spark.read.parquet(file2)
df3=spark.read.parquet(file3)
df4=spark.read.parquet(file4)

df1.show()
df2.show()
df3.show(truncate=False)
df4.show(truncate=False)

+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|    order_timestamp|    status|total_amount|
+--------+-----------+-------------------+----------+------------+
|    1001|         32|2026-08-20 00:34:25|PROCESSING|     1907.46|
+--------+-----------+-------------------+----------+------------+

+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|    order_timestamp|    status|total_amount|
+--------+-----------+-------------------+----------+------------+
|    1001|       5501|2026-09-10 08:15:00|PROCESSING|      120.50|
+--------+-----------+-------------------+----------+------------+

+----------------------------------------------------------------------------------------------------------------------------------+---+
|file_path                                                                                                                         |pos|
+-----------------------------------------------------

In [14]:
%%sql
SELECT * FROM demo.ecommerce.orders.manifests;               -- manifest list entries
        -- manifests across all snapshots

+-------+--------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------------+
|content|path                                                                                  |length|partition_spec_id|added_snapshot_id |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries                     |
+-------+--------------------------------------------------------------------------------------+------+-----------------+------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------

## EXPLANATION of the UPDATE OPERATION AND then 2 mainifest file creation and 4 parquet file creation

There are **4 Parquet files total** (2 data files + 2 delete files) and **2 manifest files** because of three things happening together: **duplicate `order_id = 1001` rows across two partitions**, **Merge-on-Read (MoR) update mechanics**, and **Iceberg v2's strict separation of data vs. delete manifests**.

First, a quick clarification on the summary metrics: you do **not** have 4 delete files. In the `summary` map, `added-delete-files -> 2` is the **total** of all delete files (positional + equality). Since `total-equality-deletes -> 0`, both of those delete files are positional delete files—which is why `added-position-delete-files -> 2` also shows `2`.

---

## 1. Why an `UPDATE` Creates Both Data Files and Delete Files (Merge-on-Read)

Parquet files are **immutable**—Iceberg cannot modify a row in-place. Because your table is using **Merge-on-Read (MoR)** for updates (`write.update.mode = merge-on-read`), Iceberg executes an `UPDATE` as a **logical DELETE + INSERT**:

* **Position Delete file (`*-deletes.parquet`):** Records the exact `(file_path, row_position)` of the old row so readers skip it at query time.
* **New Data file (`*.parquet`):** Writes the new version of the row with `status = 'PROCESSING'`.

This also explains why your snapshot summary shows `total-records -> 1005` (up from `1003` in the previous snapshot) alongside `total-position-deletes -> 2`. Physically, the data files now hold 1,005 rows, and readers subtract the 2 position-deleted rows on the fly to return **1,003 live rows**.

---

## 2. Why 4 Parquet Files Were Created Instead of 2

Your `WHERE order_id = 1001` predicate matched **2 rows**, not 1—and those 2 rows live in **2 different partitions** (`changed-partition-count -> 2`).

Look at the `partition` and `record_count` columns from your manifest output:

* `order_timestamp_day=2026-08-20` → `record_count = 1`
* `order_timestamp_day=2026-09-10` → `record_count = 1`

Because Iceberg does not enforce primary-key uniqueness on `order_id`, `order_id = 1001` exists in both partitions (likely once in the initial 1,000-row load and once in the 3-row append). Since **a single Parquet file cannot span multiple partitions**, Spark had to write a separate pair of files for each partition:

| Partition | New Data File | Position Delete File |
| --- | --- | --- |
| **`2026-08-20`** | `...00002.parquet` (1 updated row) | `...00001-deletes.parquet` (1 deleted pos) |
| **`2026-09-10`** | `...00001.parquet` (1 updated row) | `...00002-deletes.parquet` (1 deleted pos) |

**2 partitions × (1 data file + 1 delete file) = 4 Parquet files.**

---

## 3. Why 2 Manifest Files (`-m0.avro` and `-m1.avro`) Were Created

In Apache Iceberg v2, **data files and delete files are never mixed inside the same manifest file**. Every manifest has a `content` type (which you can see in your Cell `[52]` output):

* **`7f590999-...-m0.avro` (`content = 0` — Data Manifest):** Tracks only the **2 new data files**.
* **`7f590999-...-m1.avro` (`content = 1` — Delete Manifest):** Tracks only the **2 new position-delete files**. (This is why `added_data_files_count` shows `0` for `m1.avro` in Cell `[52]`—its count lives in `added_delete_files_count` instead.)

Both manifests are then registered inside your latest snapshot's manifest list (`snap-3281998439897304526-...avro`), alongside the older data manifests carried forward from snapshots `5110494612978070953` and `575426542183088885`.

In [9]:
%%sql
SELECT * FROM demo.ecommerce.orders.files;                   -- every data file + stats

-- This gives information of the data file and the partitions that it belongs to and the lower and upper bound of the partition column info.


+-------+----------------------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------+------------------+----------------------------------------------------+--------------------------------------------------+----------------------------------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------+------------+-------------+--------------------+--------------+---------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content|file_

In [10]:
%%sql
SELECT * FROM demo.ecommerce.orders.partitions;              -- partition-level stats
        -- manifests across all snapshots

+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition   |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{2026-07-15}|0      |318         |1         |5242                         |0                           |0                         |0                           |0                         |2026-11-24 01:45:28.342|575426542183088885      |
|{2026-08-20}|0      |324         |1         |53

In [11]:
%%sql

SELECT * FROM demo.ecommerce.orders.metadata_log_entries;    -- metadata.json version history
  

+-----------------------+--------------------------------------------------------------------------------------------------+------------------+----------------+----------------------+
|timestamp              |file                                                                                              |latest_snapshot_id|latest_schema_id|latest_sequence_number|
+-----------------------+--------------------------------------------------------------------------------------------------+------------------+----------------+----------------------+
|2026-11-24 01:42:57.03 |s3a://warehouse/ecommerce/orders/metadata/00000-af755d8a-1ee0-45b4-8af9-8444b3724642.metadata.json|NULL              |NULL            |NULL                  |
|2026-11-24 01:45:28.342|s3a://warehouse/ecommerce/orders/metadata/00001-0406f69a-07ec-4668-8ec9-13c53679744d.metadata.json|575426542183088885|0               |1                     |
+-----------------------+-------------------------------------------------------

In [12]:
%%sql

SELECT * FROM demo.ecommerce.orders.refs;                    -- branches & tags


+----+------+------------------+-----------------------+---------------------+----------------------+
|name|type  |snapshot_id       |max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+----+------+------------------+-----------------------+---------------------+----------------------+
|main|BRANCH|575426542183088885|NULL                   |NULL                 |NULL                  |
+----+------+------------------+-----------------------+---------------------+----------------------+



In [13]:
%%sql

SELECT * FROM demo.ecommerce.orders.all_data_files;          -- data files across all snapshots
SELECT * FROM demo.ecommerce.orders.all_manifests; 

+-------+----------------------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------+------------------+----------------------------------------------------+--------------------------------------------------+----------------------------------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------+------------+-------------+--------------------+--------------+---------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content|file_

In [17]:
# Define the full S3A path to your MinIO file

## MANIFEST LIST
manifest_list_file_path = "s3a://warehouse/ecommerce/orders/metadata/snap-575426542183088885-1-ccfe379b-a3e3-4c5d-bb4a-aa693ec8eae2.avro"

# Read the Avro file into a DataFrame
df = spark.read.format("avro").load(manifest_list_file_path)
# [span_1](start_span)[span_1](end_span)

# Print the schema (this displays all column names and data types)
df.printSchema()
# [span_2](start_span)[span_2](end_span)

# Print the output data (headers are included automatically)
print("-----------------Printing ALL Columns---------------------")
df.show(truncate=False)

print("-----------------Printing Selected Columns---------------------")

# df.select("manifest_path","manifest_length","sequence_number","added_snapshot_id","added_files_count","deleted_files_count","added_rows_count",
#           "deleted_rows_count","partitions.contains_null","partitions.lower_bound","partitions.upper_bound").orderBy("sequence_number").show(truncate=False)

df.select("manifest_path","sequence_number","added_snapshot_id","added_files_count","deleted_files_count","added_rows_count",
          "deleted_rows_count","partitions.contains_null","partitions.lower_bound","partitions.upper_bound").orderBy("sequence_number").show(truncate=False)

# [span_3](start_span)[span_3](end_span)


root
 |-- manifest_path: string (nullable = true)
 |-- manifest_length: long (nullable = true)
 |-- partition_spec_id: integer (nullable = true)
 |-- content: integer (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- min_sequence_number: long (nullable = true)
 |-- added_snapshot_id: long (nullable = true)
 |-- added_files_count: integer (nullable = true)
 |-- existing_files_count: integer (nullable = true)
 |-- deleted_files_count: integer (nullable = true)
 |-- added_rows_count: long (nullable = true)
 |-- existing_rows_count: long (nullable = true)
 |-- deleted_rows_count: long (nullable = true)
 |-- partitions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- contains_null: boolean (nullable = true)
 |    |    |-- contains_nan: boolean (nullable = true)
 |    |    |-- lower_bound: binary (nullable = true)
 |    |    |-- upper_bound: binary (nullable = true)
 |-- key_metadata: binary (nullable = true)

+------------------------

## EXPLAINATION of LOWER and UPPER BOUND
In your notebook, lower_bound and upper_bound in the manifest list (partitions.lower_bound / partitions.upper_bound) are stored as raw binary ([A9 50 00 00], [E2 50 00 00], [82 4E 00 00], [84 4E 00 00]), representing little-endian hexadecimal byte arrays of Apache Iceberg's serialized partition values—not plain human-readable dates or strings.

1. Why They Look Like Hex Bytes ([A9 50 00 00])
Binary encoding in Iceberg metadata: Inside Iceberg's manifest list Avro file, partitions is an array of partition summary structs (contains_null, contains_nan, lower_bound, upper_bound). Iceberg serializes partition values into binary buffers according to the partition field's underlying type.

Little-endian byte order: Multi-byte numbers in these hex arrays are typically stored in little-endian order (least significant byte first). For example, [A9 50 00 00] has bytes A9 (low byte), 50, 00, 00 (high byte), which reverses to big-endian 0x000050A9.

2. How to Interpret Them in Your Two Examples
Example A: ecommerce.orders ([A9 50 00 00] to [E2 50 00 00])
Look at the .files and .all_manifests outputs in your notebook for ecommerce.orders: the data files are partitioned by order_timestamp_day, spanning 2026-07-15 to 2026-09-10.

In Iceberg's days(...) partition transform, a day value is stored internally as an integer count of days since the Unix epoch (1970-01-01) (a 32-bit / 4-byte integer).

Converting little-endian hex to decimal days since 1970-01-01:

[A9 50 00 00] → big-endian 0x000050A9 = 20,649 days after 1970-01-01 → 2026-07-15.

[E2 50 00 00] → big-endian 0x000050E2 = 20,706 days after 1970-01-01 → 2026-09-10.


Here is a detailed breakdown of the `metadata.json` snippet you shared, explained in the context of your specific `trips` table scenario, followed by a clarification of the manifest files.

### 1. Breakdown of the Snapshot JSON Parameters

This JSON represents **Snapshot 3**, which was created when you ran a `DELETE` (or `UPDATE`) SQL query on your `trips` table.

* **`sequence-number: 3`**: This is the 3rd modification made to the table. (1 was the initial insert, 2 was likely an update/insert, and 3 is this current delete/update operation).
* **`snapshot-id: 3529268444000415922`**: The unique identifier for this specific point-in-time state of your table.
* **`parent-snapshot-id: 9030272551482834429`**: This points back to Snapshot 2. It tells Iceberg, "To understand the full table, start with Snapshot 2, and then apply the changes I am about to describe."
* **`summary.operation: "delete"`**: The Spark SQL operation that triggered this snapshot resulted in deleting rows (in a Merge-On-Read table, updates are often implemented as a delete + an insert).
* **`summary.added-position-delete-files: "1"` & `added-delete-files: "1"**`: Your operation created **1 new** Parquet delete file.
* **`summary.removed-position-delete-files: "1"` & `removed-delete-files: "1"**`: Iceberg actively marked **1 older** Parquet delete file as obsolete/removed. *(Note: Look closely at the JSON you pasted above—these exact lines are indeed in your snippet!)*
* **`summary.added-position-deletes: "2"`**: The newly created delete file contains the physical locations of **2 specific rows** that should be ignored when reading the data.
* **`summary.removed-position-deletes: "1"`**: The old delete file that was removed contained only **1 row** deletion.
* **`summary.total-data-files: "4"` & `total-records: "6"**`: Because this is a Merge-On-Read table, Iceberg did not delete or modify your actual base data files (`.parquet`). You still have 4 data files containing 6 total historical rows on disk. The reader will just use the delete files to filter out the 2 deleted rows on the fly.
* **`summary.total-delete-files: "1"`**: Since Iceberg added 1 delete file and removed 1 delete file in this snapshot, the net total of *active* delete files for the entire table remains 1.

---

### 2. Clarifying the Two Manifest Files (`-m1.avro` and `-m0.avro`)

To understand why Iceberg generated two separate manifest files for Snapshot 3, look at the dataframe output from **Cell `[42]**` in your notebook screenshot.

When you ran your SQL query, Iceberg realized it needed to track 2 deleted rows for the `2025-01-11` partition. Instead of keeping the old delete file (which only had 1 row) and writing a second small delete file, Iceberg combined them. It wrote a brand new delete file containing *both* deleted rows.

Because manifest files are strictly **immutable** (they cannot be edited once written), Iceberg had to write two new manifest files to record this swap:

1. **The `-m1.avro` Manifest (The Addition):**
* If you look at the 3rd dataframe in Cell `[42]`, it shows `status=1` (ADDED).
* This manifest introduces the newly created delete file (`...-00001-deletes.parquet`, size `1687` bytes).
* This file contains the 2 deleted rows (`added-position-deletes: 2`).


2. **The `-m0.avro` Manifest (The Tombstone/Deletion):**
* If you look at the 4th dataframe in Cell `[42]`, it shows `status=2` (DELETED).
* This manifest acts as a tombstone. It tells Iceberg to stop using the old delete file from Snapshot 2 (`size: 1636` bytes, which contained 1 deleted row).
* Even though the physical file might still exist on MinIO until you run a `VACUUM` or expire snapshots, this manifest tells the Iceberg reader to completely ignore it.



So, the `manifest-list` for Snapshot 3 points to both of these files so the reader knows exactly which new file to use and which old file to ignore.

In [31]:
parquet_data_file="s3a://warehouse/local/urbanfleet/trips/metadata/d2be8abb-e79e-488b-a9a8-57a848a3d80f-m0.avro"

data_df= spark.read.format(parquet_data_file)

data_df.show(truncate=False)

26/10/10 13:58:23 ERROR Executor: Exception in task 0.0 in stage 28.0 (TID 28)
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:493)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:485)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.scala:80)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:858)
	at org.apache.spar

Py4JJavaError: An error occurred while calling o289.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 28.0 failed 1 times, most recent failure: Lost task 0.0 in stage 28.0 (TID 28) (772b89eb2a63 executor driver): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:493)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:485)
	at org.apache.spark.sql.execution.d

In [19]:
manifest_file_1="s3a://warehouse/local/urbanfleet/trips/metadata/afb2b273-41da-4581-8227-9c8ace235b81-m0.avro"
df = spark.read.format("avro").load(manifest_file_1)

df.printSchema()
# [span_2](start_span)[span_2](end_span)

# Print the output data (headers are included automatically)
df.orderBy("sequence_number").show(truncate=False)



root
 |-- status: integer (nullable = true)
 |-- snapshot_id: long (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- file_sequence_number: long (nullable = true)
 |-- data_file: struct (nullable = true)
 |    |-- content: integer (nullable = true)
 |    |-- file_path: string (nullable = true)
 |    |-- file_format: string (nullable = true)
 |    |-- partition: struct (nullable = true)
 |    |    |-- trip_timestamp_day: date (nullable = true)
 |    |-- record_count: long (nullable = true)
 |    |-- file_size_in_bytes: long (nullable = true)
 |    |-- column_sizes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- key: integer (nullable = true)
 |    |    |    |-- value: long (nullable = true)
 |    |-- value_counts: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- key: integer (nullable = true)
 |    |    |    |-- value: long (nullable = true)
 |    |-- null_value_counts: arr

In [ ]:
delete_file1= "s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-7-18da0a2e-0769-4b51-8224-33c604a0ad73-00001-deletes.parquet"

df=spark.read.parquet(delete_file1)

df.show(truncate=False)

delete_file2= "s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-11-bced53f4-866e-4cec-b2ea-800bdd0867a1-00001-deletes.parquet"
df=spark.read.parquet(delete_file2)

df.show(truncate=False)


In [2]:
# 2. Create a Namespace (Schema)
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.training")
print("Namespace 'demo.training' ready.")

# 3. Create an Iceberg Table
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.training.employee (
        emp_id BIGINT,
        emp_name STRING,
        department STRING,
        salary DOUBLE
    ) USING iceberg
""")
print("Iceberg table 'employee' ready.")

26/09/13 16:25:58 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


Namespace 'demo.training' ready.
Iceberg table 'employee' ready.


In [3]:
spark.sql("Select * from demo.training.employee")

DataFrame[emp_id: bigint, emp_name: string, department: string, salary: double]

In [4]:
# 4. Write Data into the Table
spark.sql("""
    INSERT INTO demo.training.employee 
    VALUES 
        (1, 'Ashish Singh', 'Data Engineering', 150000.00),
        (2, 'Preeti Singh', 'Analytics', 135000.00)
""")
print("Data successfully committed to MinIO.")

# 5. Query the Active Data
print("\n--- CURRENT TABLE DATA ---")
spark.sql("SELECT * FROM demo.training.employee").show()

Data successfully committed to MinIO.

--- CURRENT TABLE DATA ---
+------+------------+----------------+--------+
|emp_id|    emp_name|      department|  salary|
+------+------------+----------------+--------+
|     1|Ashish Singh|Data Engineering|150000.0|
|     2|Preeti Singh|       Analytics|135000.0|
+------+------------+----------------+--------+



In [5]:

# ==========================================
# 6. VISUALIZE ICEBERG METADATA & HISTORY
# ==========================================

print("\n--- TABLE HISTORY (Timeline of Commits) ---")
# Shows when the table was created and every subsequent commit timestamp
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

print("\n--- TABLE SNAPSHOTS ---")
# Shows the specific operations (e.g., 'append', 'overwrite') and row counts for each commit
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

print("\n--- PHYSICAL DATA FILES ---")
# Exposes the actual Parquet files sitting in your MinIO 'warehouse' bucket
spark.sql("SELECT file_path, file_size_in_bytes, record_count FROM demo.training.employee.files").show(truncate=False)


--- TABLE HISTORY (Timeline of Commits) ---
+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-09-13 16:26:28.906|7001475892194027695|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+


--- TABLE SNAPSHOTS ---
+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                     

In [6]:
# Create a DataFrame
data = [(3, 'Ravi Kumar', 'Cloud Infrastructure', 1400200.00)]
columns = ["emp_id", "emp_name", "department", "salary"]
df = spark.createDataFrame(data, columns)

# Write to Iceberg using the V2 API
df.writeTo("demo.training.employee").append()

# Run the snapshots query again to see the new 'append' operation logged
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------

In [6]:
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-03 17:02:53.083|2144457049189745365|NULL               |true               |
|2026-09-03 17:03:01.438|3568146871021356834|2144457049189745365|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [7]:
%%sql
    
SELECT * FROM demo.training.employee.history

26/09/03 17:03:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


made_current_at,snapshot_id,parent_id,is_current_ancestor
2026-09-03 17:02:53.083000,2144457049189745365,None,True
2026-09-03 17:03:01.438000,3568146871021356834,2144457049189745365,True


In [8]:
%%sql
SELECT * FROM demo.training.employee version as of 2144457049189745365;


emp_id,emp_name,department,salary
1,Ashish Singh,Data Engineering,150000.0
2,Preeti Singh,Analytics,135000.0
